# LOB Simulation Live Demos (Teaching Notebook)

"
"This notebook gives **9 live demos** for key functions in `LOB_Simulation.py`.

"
"Sign convention used throughout:
"
"- `qty > 0` means **BUY**
"
"- `qty < 0` means **SELL**

"
"Reproducibility rule:
"
"- Every demo creates a **fresh `OrderBook`** with fixed parameters and seed, so outputs are deterministic across reruns.


In [3]:
from pprint import pprint
from LOB_Simulation import OrderBook

# Helper to print a compact top-of-book summary.
def print_top_of_book(book, label="Top of Book"):
    best_bid = book.bids[0] if book.bids else (None, None)
    best_ask = book.asks[0] if book.asks else (None, None)
    print(f"{label}")
    print(f"  Best Bid: {best_bid[0]} x {best_bid[1]}")
    print(f"  Best Ask: {best_ask[0]} x {best_ask[1]}")


In [4]:
# Baseline deterministic parameters used in all demos.
REFERENCE_PRICE = 100.0
TICK_SIZE = 0.01
N_LEVELS = 8
SEED = 123

# Factory helper so each cell starts from a fresh independent book state.
def fresh_book():
    return OrderBook(
        reference_price=REFERENCE_PRICE,
        tick_size=TICK_SIZE,
        n_levels=N_LEVELS,
        seed=SEED,
    )

print("Baseline parameters set:")
print(f"reference_price={REFERENCE_PRICE}, tick_size={TICK_SIZE}, n_levels={N_LEVELS}, seed={SEED}")


Baseline parameters set:
reference_price=100.0, tick_size=0.01, n_levels=8, seed=123


In [5]:
# Demo 1: order_book_levels(...)
# This function creates the bid/ask ladders and stores target side volumes for future refill.
book = fresh_book()

execution_qty = 250
noise = 0.0
mult_low = 6.0
mult_high = 6.0
hump_center = 3
hump_sigma = 1.5
tail_decay = 0.20
top_dip = 0.40

print("Calling order_book_levels with explicit fixed inputs...")
bids, asks = book.order_book_levels(
    execution_qty=execution_qty,
    noise=noise,
    mult_low=mult_low,
    mult_high=mult_high,
    hump_center=hump_center,
    hump_sigma=hump_sigma,
    tail_decay=tail_decay,
    top_dip=top_dip,
)

best_bid = bids[0]
best_ask = asks[0]
spread = best_ask[0] - best_bid[0]

print(f"Number of bid levels: {len(bids)}")
print(f"Number of ask levels: {len(asks)}")
print(f"Best bid: {best_bid[0]} x {best_bid[1]}")
print(f"Best ask: {best_ask[0]} x {best_ask[1]}")
print(f"Spread: {spread:.2f}")
print(f"Target buy volume stored in book:  {book.target_buy_vol:.2f}")
print(f"Target sell volume stored in book: {book.target_sell_vol:.2f}")
print(f"Top 3 bids: {bids[:3]}")
print(f"Top 3 asks: {asks[:3]}")


Calling order_book_levels with explicit fixed inputs...
Number of bid levels: 8
Number of ask levels: 8
Best bid: 99.94 x 40
Best ask: 100.06 x 39
Spread: 0.12
Target buy volume stored in book:  1500.00
Target sell volume stored in book: 1472.17
Top 3 bids: [(99.94, 40), (99.93, 248), (99.92, 396)]
Top 3 asks: [(100.06, 39), (100.07000000000001, 243), (100.08, 388)]


In [6]:
# Demo 2: execute_market_order(...)
# A positive signed quantity is a BUY market order, which consumes the ask side.
book = fresh_book()
book.order_book_levels(
    execution_qty=2000,
    noise=0.10,
    mult_low=5.0,
    mult_high=10.0,
    hump_center=3,
    hump_sigma=1.5,
    tail_decay=0.20,
    top_dip=0.40,
)

print_top_of_book(book, "Before execution")

market_order_qty = 5000
report = book.execute_market_order(
    qty_signed=market_order_qty,
    impact_eta_ticks=0.0,
    impact_gamma=1.5,
    impact_scale=None,
    impact_use_cum=True,
)

print("\nExecution report summary:")
print(f"Requested qty: {report['requested_qty']}")
print(f"Filled qty:    {report['filled_qty']}")
print(f"Remaining qty: {report['remaining_qty']}")
print(f"VWAP:          {report['vwap']}")
print("Fills (price, signed_qty):")
for price, signed_qty in report["fills"]:
    print(f"  {price:.2f} x {signed_qty:+d}")

print()
print_top_of_book(book, "After execution")


Before execution
  Best Bid: 99.94 x 267
  Best Ask: 100.06 x 274

Execution report summary:
Requested qty: 5000
Filled qty:    5000
Remaining qty: 0
VWAP:          100.07658800000002
Fills (price, signed_qty):
  100.06 x +274
  100.07 x +1675
  100.08 x +2534
  100.09 x +517

After execution
  Best Bid: 99.94 x 267
  Best Ask: 100.09 x 2324


In [7]:
# Demo 3: refill_step(...)
# We first deplete the best bid with a sell market order, then refill the book.
book = fresh_book()
book.order_book_levels(execution_qty=100, noise=0.0)

print("Initial top 3 levels:")
print(f"Bids: {book.bids[:3]}")
print(f"Asks: {book.asks[:3]}")

# Deplete the best bid level exactly.
depletion_qty = book.bids[0][1]
execution = book.execute_market_order(-depletion_qty)

print("\nAfter depletion trade:")
print(f"requested_qty={execution['requested_qty']}, filled_qty={execution['filled_qty']}, remaining_qty={execution['remaining_qty']}")
print(f"last_exhausted_best_bid={book.last_exhausted_best_bid}")
print(f"Top 3 bids before refill: {book.bids[:3]}")

# Refill deterministically (noise=0.0).
book.refill_step(rho=0.35, noise=0.0)

print("\nAfter refill_step(rho=0.35, noise=0.0):")
print(f"Top 3 bids: {book.bids[:3]}")
print(f"Top 3 asks: {book.asks[:3]}")


Initial top 3 levels:
Bids: [(99.94, 14), (99.93, 90), (99.92, 143)]
Asks: [(100.06, 14), (100.07000000000001, 88), (100.08, 141)]

After depletion trade:
requested_qty=-14, filled_qty=-14, remaining_qty=0
last_exhausted_best_bid=99.94
Top 3 bids before refill: [(99.93, 90), (99.92, 143), (99.91, 147)]

After refill_step(rho=0.35, noise=0.0):
Top 3 bids: [(99.93, 64), (99.92, 124), (99.91, 146)]
Top 3 asks: [(100.06, 14), (100.07000000000001, 88), (100.08, 141)]


In [9]:
# Demo 4: step(...)
# One full simulation step can include main trade(s), background flow, permanent shift, MM restore, and refill.
book = fresh_book()
book.order_book_levels(
    execution_qty=180,
    noise=0.0,
    mult_low=6.0,
    mult_high=6.0,
    hump_center=2,
    hump_sigma=1.0,
    tail_decay=0.15,
    top_dip=0.35,
)
book.mm_p_restore = 0.0  # Keep this demo focused on step mechanics.

print_top_of_book(book, "Before step")

log = book.step(
    t=1,
    trades=[-180],
    bg_lam=0.0,
    bg_p_buy=0.5,
    rho=0.25,
    noise=0.0,
    perm_eta_ticks=0.6,
    perm_gamma=1.0,
    perm_scale=7500,
    main_exec_kwargs={"impact_eta_ticks": 0.0, "impact_gamma": 1.0, "impact_scale": None, "impact_use_cum": True},
    bg_exec_kwargs={"impact_eta_ticks": 0.0, "impact_gamma": 1.0, "impact_scale": None, "impact_use_cum": True},
    order="main_first",
)

print("\nReturned log keys:", list(log.keys()))
print(f"t={log['t']}")
print(f"net_main_filled={log['net_main_filled']}")
print(f"delta_p_perm={log['delta_p_perm']}")
print(f"bg_orders={log['bg_orders']}")
print(f"trade_reports_count={len(log['trade_reports'])}")
if log["trade_reports"]:
    tr = log["trade_reports"][0]
    print("Main trade report summary:")
    print(f"  requested={tr['requested_qty']}, filled={tr['filled_qty']}, remaining={tr['remaining_qty']}, vwap={tr['vwap']}")

print()
print_top_of_book(book, "After step")


Before step
  Best Bid: 99.94 x 29
  Best Ask: 100.06 x 28

Returned log keys: ['t', 'bg_orders', 'bg_reports', 'trade_reports', 'net_main_filled', 'delta_p_perm']
t=1
net_main_filled=-180
delta_p_perm=-0.000144
bg_orders=[]
trade_reports_count=1
Main trade report summary:
  requested=-180, filled=-180, remaining=0, vwap=99.93161111111111

After step
  Best Bid: 99.92 x 132
  Best Ask: 100.05 x 28


In [10]:
# Demo 5: run(...)
# This function executes multiple simulation steps using a trade schedule.
book = fresh_book()
book.order_book_levels(
    execution_qty=200,
    noise=0.03,
    mult_low=6.0,
    mult_high=8.0,
    hump_center=2,
    hump_sigma=1.2,
    tail_decay=0.18,
    top_dip=0.35,
)

trade_schedule = {
    1: [120],
    3: [-90, -30],
    5: [180],
}

history, step_logs = book.run(
    n_steps=5,
    trade_schedule=trade_schedule,
    bg_lam=1.0,
    bg_p_buy=0.55,
    rho=0.25,
    noise=0.03,
    perm_eta_ticks=0.6,
    perm_gamma=1.0,
    perm_scale=5000,
    main_exec_kwargs={"impact_eta_ticks": 0.4, "impact_gamma": 1.2, "impact_scale": 400},
    bg_exec_kwargs={"impact_eta_ticks": 0.15, "impact_gamma": 1.1, "impact_scale": 250},
    order="bg_first",
)

print(f"history length = {len(history)} (includes t=0 initial snapshot)")
print(f"step_logs keys = {list(step_logs.keys())}")
print(f"Initial snapshot bests: bid={history[0]['bids'][0]}, ask={history[0]['asks'][0]}")
print(f"Final snapshot bests:   bid={history[-1]['bids'][0]}, ask={history[-1]['asks'][0]}")

print("\nPer-step summary:")
for t in sorted(step_logs):
    log = step_logs[t]
    print(
        f"t={t}: bg_orders={len(log['bg_orders'])}, "
        f"main_reports={len(log['trade_reports'])}, "
        f"net_main_filled={log['net_main_filled']}, "
        f"delta_p_perm={log['delta_p_perm']:.6f}"
    )


history length = 6 (includes t=0 initial snapshot)
step_logs keys = [1, 2, 3, 4, 5]
Initial snapshot bests: bid=(99.94, 55), ask=(100.06, 54)
Final snapshot bests:   bid=(99.96000000000001, 33), ask=(100.07000000000001, 20)

Per-step summary:
t=1: bg_orders=1, main_reports=1, net_main_filled=120, delta_p_perm=0.000144
t=2: bg_orders=0, main_reports=0, net_main_filled=0, delta_p_perm=0.000000
t=3: bg_orders=0, main_reports=2, net_main_filled=-120, delta_p_perm=-0.000144
t=4: bg_orders=2, main_reports=0, net_main_filled=0, delta_p_perm=0.000000
t=5: bg_orders=1, main_reports=1, net_main_filled=180, delta_p_perm=0.000216


In [11]:
# Demo 6: sample_background_orders(...)
# This function samples random background market orders between main decisions.
book = fresh_book()

orders = book.sample_background_orders(
    lam=4.0,
    p_buy=0.60,
    size_mean=150,
    size_std=0.50,
    size_min=10,
    size_max=500,
)

print("Sampled background orders:", orders)
print("Interpretation (+ is BUY, - is SELL):")
if not orders:
    print("  No orders sampled in this run.")
else:
    for i, q in enumerate(orders, start=1):
        side = "BUY" if q > 0 else "SELL"
        print(f"  {i}. {side} {abs(q)} (signed={q})")


Sampled background orders: [165, 238]
Interpretation (+ is BUY, - is SELL):
  1. BUY 165 (signed=165)
  2. BUY 238 (signed=238)


In [12]:
# Demo 7: apply_background_flow(...)
# This function samples background orders and executes each using execute_market_order.
book = fresh_book()
book.order_book_levels(
    execution_qty=500,
    noise=0.08,
    mult_low=6.0,
    mult_high=6.0,
    hump_center=3,
    hump_sigma=1.4,
    tail_decay=0.18,
    top_dip=0.35,
)

orders, reports = book.apply_background_flow(
    lam=4.0,
    p_buy=0.55,
    exec_kwargs={
        "impact_eta_ticks": 0.25,
        "impact_gamma": 1.3,
        "impact_scale": 750.0,
        "impact_use_cum": True,
    },
    size_mean=180,
    size_std=0.6,
    size_min=25,
    size_max=600,
)

print("Sampled background orders:", orders)
print("Per-order execution summaries:")
for i, report in enumerate(reports, start=1):
    vwap = report["vwap"]
    vwap_text = f"{vwap:.4f}" if vwap is not None else "None"
    print(
        f"  Report {i}: requested={report['requested_qty']}, "
        f"filled={report['filled_qty']}, vwap={vwap_text}, fills={len(report['fills'])}"
    )


Sampled background orders: [202, -313]
Per-order execution summaries:
  Report 1: requested=202, filled=202, vwap=100.0775, fills=2
  Report 2: requested=-313, filled=-313, vwap=99.9216, fills=2


In [13]:
# Demo 8: shift_book_prices(...)
# This applies a permanent price shift to both sides of the book.
book = fresh_book()
book.order_book_levels(execution_qty=1000)

before_best_bid = book.bids[0][0]
before_best_ask = book.asks[0][0]
print("Before shift:")
print(f"  Best bid={before_best_bid:.2f}, Best ask={before_best_ask:.2f}")
print(f"  Top 3 bids={[(f'{p:.2f}', q) for p, q in book.bids[:3]]}")
print(f"  Top 3 asks={[(f'{p:.2f}', q) for p, q in book.asks[:3]]}")

book.shift_book_prices(delta_p=0.03)  # 3 ticks upward with tick size 0.01

after_best_bid = book.bids[0][0]
after_best_ask = book.asks[0][0]
print("\nAfter shift_book_prices(delta_p=0.03):")
print(f"  Best bid={after_best_bid:.2f}, Best ask={after_best_ask:.2f}")
print(f"  Top 3 bids={[(f'{p:.2f}', q) for p, q in book.bids[:3]]}")
print(f"  Top 3 asks={[(f'{p:.2f}', q) for p, q in book.asks[:3]]}")


Before shift:
  Best bid=99.94, Best ask=100.06
  Top 3 bids=[('99.94', 133), ('99.93', 971), ('99.92', 1301)]
  Top 3 asks=[('100.06', 137), ('100.07', 837), ('100.08', 1267)]

After shift_book_prices(delta_p=0.03):
  Best bid=99.97, Best ask=100.09
  Top 3 bids=[('99.97', 133), ('99.96', 971), ('99.95', 1301)]
  Top 3 asks=[('100.09', 137), ('100.10', 837), ('100.11', 1267)]


In [14]:
# Demo 9: mm_restore_exhausted_top(...)
# If the top level was exhausted, MM may restore it. We force restoration with p_restore=1.0.
book = fresh_book()
book.order_book_levels(execution_qty=100, noise=0.0)

print_top_of_book(book, "Initial top of book")

# Exhaust best ask exactly using a BUY market order.
exhausted_ask_px, exhausted_ask_qty = book.asks[0]
buy_report = book.execute_market_order(exhausted_ask_qty)

print("\nAfter exhausting top ask with one buy order:")
print(f"  Exhausted ask price={exhausted_ask_px:.2f}, consumed qty={exhausted_ask_qty}")
print(f"  Filled qty from report={buy_report['filled_qty']}")
print_top_of_book(book, "Top of book before restore")

# Expected restore quantity follows vol_frac * current best ask qty (after exhaustion), floored by min_qty.
new_best_ask_qty = book.asks[0][1]
expected_restored_qty = max(1, int(round(0.5 * new_best_ask_qty)))

book.mm_restore_exhausted_top(p_restore=1.0, vol_frac=0.5, min_qty=1)

print("\nAfter mm_restore_exhausted_top(p_restore=1.0, vol_frac=0.5, min_qty=1):")
print_top_of_book(book, "Top of book after restore")
print("Restoration check:")
print(f"  Expected restored price={exhausted_ask_px:.2f}")
print(f"  Expected restored qty={expected_restored_qty}")
print(f"  Actual restored top ask={book.asks[0]}")


Initial top of book
  Best Bid: 99.94 x 14
  Best Ask: 100.06 x 14

After exhausting top ask with one buy order:
  Exhausted ask price=100.06, consumed qty=14
  Filled qty from report=14
Top of book before restore
  Best Bid: 99.94 x 14
  Best Ask: 100.07000000000001 x 88

After mm_restore_exhausted_top(p_restore=1.0, vol_frac=0.5, min_qty=1):
Top of book after restore
  Best Bid: 99.94 x 14
  Best Ask: 100.06 x 44
Restoration check:
  Expected restored price=100.06
  Expected restored qty=44
  Actual restored top ask=(100.06, 44)
